# ECCOFS `avg`: best-estimate surface temperature from a virtual Icechunk store

The store at `https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-fmrc` on [Source Cooperative](https://source.coop) is a virtual Icechunk repository over the NOAA ECCOFS `avg` NetCDF files.
It holds one entry per forecast run: `time` is the run's reference time and `step` is the lead day. Valid times therefore overlap between runs.

This notebook:
1. opens the store from Source Cooperative,
2. uses [Rolodex](https://github.com/dcherian/rolodex) to assemble a monotonic, non-duplicating best-estimate time series,
3. picks the surface level of `temp`,
4. plots it with hvPlot.

(The sibling store `eccofs-avg-best` already holds this best-estimate series; see `eccofs_avg_best_series_surface_temp.ipynb`.)

No credentials are needed: both Source Cooperative (which holds the store) and NOAA's bucket (which holds the NetCDF bytes) are read anonymously. It needs icechunk 2.x, which needs Python 3.12 or newer.

In [ ]:
import hvplot.xarray  # noqa: F401
import icechunk
import numpy as np
import rolodex.forecast
import xarray as xr
import zarr
from rolodex.forecast import BestEstimate, ForecastIndex

zarr.config.set({"async.concurrency": 32})

URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-fmrc"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

In [ ]:
repo = icechunk.Repository.open(icechunk.http_storage(URL))
# the NetCDF bytes live in NOAA's bucket, outside the store: authorize that virtual chunk container (anonymous)
repo = repo.reopen(authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}))
latest = next(iter(repo.ancestry(branch="main")))
print(latest.written_at, "|", latest.message)

ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
ds[["temp"]]

## Best-estimate time series with Rolodex

`valid_time = time + step` is not stored, so Rolodex builds it lazily and indexes `(time, step, valid_time)` together.

`BestEstimate(offset=1)` takes each valid time from the most recent run in which it is at least one lead day in. That follows the modeler's recommendation for `avg`:
- the middle day of each 3-day analysis, which is file suffix `_0002` (`step` = 1 day), for all past times, to avoid start-of-cycle transients;
- then the rest of the latest run (`_0003` to `_0008`) as the forecast.

If a run is missing, Rolodex falls back to the next most recent run that covers that valid time.

In [ ]:
fc = ds[["temp"]]
fc.coords["valid_time"] = rolodex.forecast.create_lazy_valid_time_variable(
    reference_time=fc.time, period=fc.step
)
fc = fc.drop_indexes(["time", "step"]).set_xindex(["time", "step", "valid_time"], ForecastIndex)

best = fc.sel(valid_time=BestEstimate(offset=1))
vt = best.valid_time.to_index()
assert vt.is_monotonic_increasing and vt.is_unique, "best-estimate valid times must be strictly increasing"
print(f"{len(vt)} valid times: {vt[0]} to {vt[-1]}")
best

## Surface layer of `temp`

ROMS orders `s_rho` from the bottom (-1) to the surface (0), so the surface is the last level. The assertion below guards against that order being reversed.

In [ ]:
assert float(best.s_rho[-1]) > float(best.s_rho[0]), "expected s_rho to increase towards the surface"
sst = best.temp.isel(s_rho=-1).rename("sst")
sst.attrs.update(long_name="surface potential temperature", units="degC")
print(f"s_rho[-1] = {float(best.s_rho[-1]):.4f} (surface); one level per chunk, so only the surface level is read")
sst

## Plot with hvPlot

The grid is curvilinear (`lon_rho`, `lat_rho`), so this uses `quadmesh` with datashader rasterization. Reads one surface level from NOAA's public bucket.

In [ ]:
%%time
t = best['temp'][0,-1,:,:].load()

In [ ]:
t.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, tiles='OSM',
    cmap="turbo", geo=True, coastline=True, frame_width=700, title="ECCOFS best-estimate surface temperature (degC)",
)